# Disruption Prediction on FAIR-MAST Tokamak Data

**FAIR-MAST** (Findable, Accessible, Interoperable, Reusable — Mega Ampere Spherical Tokamak) is an open archive of plasma physics experiments from the MAST tokamak at UKAEA, UK. It contains over 11,500 shots with time-series diagnostics: plasma current, magnetic fields, electron density, temperature profiles, and more.

**Plasma disruptions** are sudden, uncontrolled losses of plasma confinement in tokamaks. They cause rapid energy dumps that can damage reactor components. Predicting disruptions in advance (even 10-30 ms) allows control systems to mitigate damage — making disruption prediction a critical ML task for fusion energy.

In this notebook we compare three ML approaches:
1. **Random Forest** — a classical baseline on flattened features
2. **Bi-LSTM + Attention** — captures temporal patterns with learnable focus
3. **Transformer Encoder** — self-attention over the full sequence

We use synthetic data that mimics the FAIR-MAST structure (500 shots x 200 timesteps x 20 features). For real results on actual FAIR-MAST data, see our platform and paper.

**Links:**
- GitHub: [sakiselev-ai/tokamak-analysis](https://github.com/sakiselev-ai/tokamak-analysis)
- Platform: [tokamak-ai.ru](https://tokamak-ai.ru)
- FAIR-MAST archive: [mast.ukaea.uk](https://mast.ukaea.uk)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Data Loading — Synthetic FAIR-MAST-like Data

Since Kaggle cannot access the FAIR-MAST S3 archive directly, we generate synthetic data that mimics the real dataset structure:
- **500 shots** (plasma experiments)
- **200 timesteps** per shot (sampled at ~1 kHz)
- **20 diagnostic features** (plasma current, loop voltage, density, magnetics, etc.)
- **Binary labels**: 0 = stable, 1 = disrupted (~30% disruption rate)

Disrupted shots have a characteristic signature: signals become unstable in the last ~30% of the time series, with growing oscillations and a sharp current drop.

In [ ]:
N_SAMPLES = 500
N_TIMESTEPS = 200
N_FEATURES = 20
DISRUPTION_RATE = 0.30

def generate_synthetic_fair_mast(n_samples, n_timesteps, n_features, disruption_rate, seed=42):
    """Generate synthetic tokamak data mimicking FAIR-MAST structure.
    
    Stable shots: smooth ramp-up, flat-top, ramp-down with mild noise.
    Disrupted shots: similar start, but develop growing oscillations
    and a sharp current drop in the final phase.
    """
    rng = np.random.RandomState(seed)
    n_disrupted = int(n_samples * disruption_rate)
    n_stable = n_samples - n_disrupted
    
    t = np.linspace(0, 1, n_timesteps)
    data = np.zeros((n_samples, n_timesteps, n_features))
    labels = np.zeros(n_samples, dtype=np.int64)
    
    # Base signal: ramp-up -> flat-top -> ramp-down
    base_profile = np.where(t < 0.2, t / 0.2,
                   np.where(t < 0.7, 1.0, 1.0 - (t - 0.7) / 0.3))
    
    for i in range(n_samples):
        is_disrupted = i < n_disrupted
        labels[i] = 1 if is_disrupted else 0
        
        for f in range(n_features):
            # Each feature has a slightly different amplitude and phase
            amp = 0.5 + rng.random() * 1.5
            phase = rng.random() * 0.1
            noise_level = 0.02 + rng.random() * 0.03
            
            signal = amp * np.roll(base_profile, int(phase * n_timesteps))
            signal += rng.normal(0, noise_level, n_timesteps)
            
            if is_disrupted:
                # Add pre-disruption signatures
                disruption_onset = int(0.5 * n_timesteps + rng.randint(-20, 20))
                disruption_time = int(0.75 * n_timesteps + rng.randint(-10, 10))
                
                # Growing oscillations before disruption
                osc_envelope = np.zeros(n_timesteps)
                osc_region = slice(disruption_onset, disruption_time)
                n_osc = disruption_time - disruption_onset
                if n_osc > 0:
                    osc_envelope[osc_region] = np.linspace(0, 0.3 * amp, n_osc)
                freq = 5 + rng.random() * 10
                signal += osc_envelope * np.sin(2 * np.pi * freq * t)
                
                # Sharp drop at disruption
                if disruption_time < n_timesteps:
                    drop_speed = 3 + rng.random() * 5
                    drop = np.zeros(n_timesteps)
                    remaining = n_timesteps - disruption_time
                    drop[disruption_time:] = -amp * (1 - np.exp(-drop_speed * np.linspace(0, 1, remaining)))
                    signal += drop
            
            data[i, :, f] = signal
    
    # Shuffle
    perm = rng.permutation(n_samples)
    data = data[perm]
    labels = labels[perm]
    
    return data, labels

data, labels = generate_synthetic_fair_mast(N_SAMPLES, N_TIMESTEPS, N_FEATURES, DISRUPTION_RATE)

print(f'Data shape:   {data.shape}  (samples, timesteps, features)')
print(f'Labels shape: {labels.shape}')
print(f'Class distribution:')
print(f'  Stable (0):    {(labels == 0).sum()} ({(labels == 0).mean():.1%})')
print(f'  Disrupted (1): {(labels == 1).sum()} ({(labels == 1).mean():.1%})')

## 2. Exploratory Data Analysis

Let's visualize typical stable and disrupted shots to understand the signal patterns our models need to learn.

In [ ]:
stable_idx = np.where(labels == 0)[0][:3]
disrupted_idx = np.where(labels == 1)[0][:3]

feature_names = [
    'Ip (plasma current)', 'Vloop (loop voltage)', 'ne (density)',
    'Te (electron temp)', 'Ti (ion temp)', 'Wmhd (stored energy)',
    'betap', 'li (inductance)', 'q95', 'Btor',
    'PNBI (NBI power)', 'Pohm (ohmic power)', 'Prad (radiated)',
    'Dalpha', 'n=1 mode', 'n=2 mode', 'elongation', 'triangularity',
    'Zaxis', 'Rgeo'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
t = np.arange(N_TIMESTEPS)

# Plot 3 features for stable vs disrupted
plot_features = [0, 2, 5]  # Ip, ne, Wmhd

for col, fidx in enumerate(plot_features):
    # Stable
    ax = axes[0, col]
    for s in stable_idx:
        ax.plot(t, data[s, :, fidx], alpha=0.7)
    ax.set_title(f'Stable — {feature_names[fidx]}')
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Signal')
    ax.grid(True, alpha=0.3)
    
    # Disrupted
    ax = axes[1, col]
    for s in disrupted_idx:
        ax.plot(t, data[s, :, fidx], alpha=0.7)
    ax.set_title(f'Disrupted — {feature_names[fidx]}')
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Signal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Stable vs Disrupted Shot Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Signal statistics
print('\nSignal Statistics (mean +/- std across all samples):')
print(f'{"Feature":<25} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10}')
print('-' * 67)
for f in range(min(10, N_FEATURES)):
    vals = data[:, :, f].flatten()
    print(f'{feature_names[f]:<25} {vals.mean():>10.4f} {vals.std():>10.4f} {vals.min():>10.4f} {vals.max():>10.4f}')

## 3. Preprocessing

- Z-score normalization (per feature, computed on training set only)
- Train / Validation / Test split: 70% / 15% / 15%

In [ ]:
# Split: 70/15/15
X_trainval, X_test, y_trainval, y_test = train_test_split(
    data, labels, test_size=0.15, random_state=SEED, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.176, random_state=SEED, stratify=y_trainval  # 0.176 ~ 15/85
)

print(f'Train: {X_train.shape[0]} samples (disrupted: {y_train.sum()}/{len(y_train)})')
print(f'Val:   {X_val.shape[0]} samples (disrupted: {y_val.sum()}/{len(y_val)})')
print(f'Test:  {X_test.shape[0]} samples (disrupted: {y_test.sum()}/{len(y_test)})')

# Z-score normalization (fit on train only)
train_mean = X_train.mean(axis=(0, 1), keepdims=True)  # (1, 1, 20)
train_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_norm = (X_train - train_mean) / train_std
X_val_norm = (X_val - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

print(f'\nAfter normalization — Train mean: {X_train_norm.mean():.6f}, std: {X_train_norm.std():.4f}')

## 4. Model 1: Random Forest Baseline

Random Forest operates on flattened features (200 timesteps x 20 features = 4000-dim vector per sample). Despite losing temporal structure, RF is a strong baseline for tabular data.

In [ ]:
# Flatten temporal data for RF
X_train_flat = X_train_norm.reshape(X_train_norm.shape[0], -1)
X_val_flat = X_val_norm.reshape(X_val_norm.shape[0], -1)
X_test_flat = X_test_norm.reshape(X_test_norm.shape[0], -1)

print(f'Flattened shape: {X_train_flat.shape}')

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train_flat, y_train)

# Evaluate
rf_pred = rf.predict(X_test_flat)
rf_proba = rf.predict_proba(X_test_flat)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

print(f'\nRandom Forest Results:')
print(f'  Accuracy: {rf_acc:.4f}')
print(f'  F1 Score: {rf_f1:.4f}')
print(f'  AUC-ROC:  {rf_auc:.4f}')
print(f'\n{classification_report(y_test, rf_pred, target_names=["Stable", "Disrupted"])}')

# Feature importances (grouped by diagnostic)
importances = rf.feature_importances_.reshape(N_TIMESTEPS, N_FEATURES)
feature_importance_avg = importances.mean(axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Per-feature importance
sorted_idx = np.argsort(feature_importance_avg)[::-1]
ax1.barh(range(N_FEATURES), feature_importance_avg[sorted_idx])
ax1.set_yticks(range(N_FEATURES))
ax1.set_yticklabels([feature_names[i] for i in sorted_idx], fontsize=8)
ax1.set_xlabel('Mean Importance')
ax1.set_title('Feature Importances (averaged over time)')
ax1.invert_yaxis()

# Temporal importance profile
temporal_importance = importances.sum(axis=1)
ax2.plot(temporal_importance)
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Total Importance')
ax2.set_title('Importance vs Time (all features summed)')
ax2.grid(True, alpha=0.3)
ax2.axvline(x=100, color='r', linestyle='--', alpha=0.5, label='Mid-shot')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Model 2: Bi-LSTM + Attention

A bidirectional LSTM processes the sequence in both directions. An attention mechanism learns to focus on the most informative timesteps (typically the pre-disruption phase). This architecture achieved **AUC 0.9938** on real FAIR-MAST data with full training.

In [ ]:
class Attention(nn.Module):
    """Additive attention over LSTM hidden states."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
    
    def forward(self, lstm_output):
        # lstm_output: (batch, seq_len, hidden_size)
        weights = torch.softmax(self.attn(lstm_output), dim=1)  # (batch, seq_len, 1)
        context = (weights * lstm_output).sum(dim=1)  # (batch, hidden_size)
        return context, weights.squeeze(-1)


class LSTMAttentionClassifier(nn.Module):
    def __init__(self, input_size=20, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = Attention(hidden_size * 2)  # *2 for bidirectional
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)  # (batch, seq_len, hidden*2)
        context, attn_weights = self.attention(lstm_out)
        logits = self.classifier(context).squeeze(-1)
        return logits, attn_weights


print('LSTM+Attention model:')
model_lstm = LSTMAttentionClassifier(input_size=N_FEATURES).to(device)
total_params = sum(p.numel() for p in model_lstm.parameters())
print(f'  Parameters: {total_params:,}')
print(model_lstm)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3):
    """Train a PyTorch binary classifier and return history."""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.BCEWithLogitsLoss()
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
    
    for epoch in range(epochs):
        # Train
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            if isinstance(logits, tuple):
                logits = logits[0]
            loss = criterion(logits, y_batch.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())
        
        # Validate
        model.eval()
        val_losses, val_preds, val_labels = [], [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                if isinstance(logits, tuple):
                    logits = logits[0]
                loss = criterion(logits, y_batch.float())
                val_losses.append(loss.item())
                val_preds.extend(torch.sigmoid(logits).cpu().numpy())
                val_labels.extend(y_batch.cpu().numpy())
        
        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        val_acc = accuracy_score(val_labels, np.array(val_preds) > 0.5)
        val_auc = roc_auc_score(val_labels, val_preds) if len(set(val_labels)) > 1 else 0.5
        
        scheduler.step(avg_val)
        
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:2d}/{epochs} — '
                  f'train_loss: {avg_train:.4f}, val_loss: {avg_val:.4f}, '
                  f'val_acc: {val_acc:.4f}, val_auc: {val_auc:.4f}')
    
    return history


def evaluate_model(model, test_loader):
    """Evaluate model on test set, return (accuracy, f1, auc, predictions, probabilities)."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            if isinstance(logits, tuple):
                logits = logits[0]
            proba = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(proba)
            all_labels.extend(y_batch.numpy())
    
    preds = np.array(all_preds)
    labels_arr = np.array(all_labels)
    binary = (preds > 0.5).astype(int)
    
    acc = accuracy_score(labels_arr, binary)
    f1 = f1_score(labels_arr, binary)
    auc = roc_auc_score(labels_arr, preds) if len(set(labels_arr)) > 1 else 0.5
    
    return acc, f1, auc


# Create DataLoaders
def make_loaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=32):
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    val_ds = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))
    test_ds = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=batch_size),
        DataLoader(test_ds, batch_size=batch_size)
    )

train_loader, val_loader, test_loader = make_loaders(
    X_train_norm, y_train, X_val_norm, y_val, X_test_norm, y_test
)

print('Training LSTM+Attention...')
lstm_history = train_model(model_lstm, train_loader, val_loader, epochs=20, lr=1e-3)

lstm_acc, lstm_f1, lstm_auc = evaluate_model(model_lstm, test_loader)
print(f'\nLSTM+Attention Test Results:')
print(f'  Accuracy: {lstm_acc:.4f}')
print(f'  F1 Score: {lstm_f1:.4f}')
print(f'  AUC-ROC:  {lstm_auc:.4f}')

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(lstm_history['train_loss'], label='Train')
ax1.plot(lstm_history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('LSTM+Attention — Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(lstm_history['val_acc'], label='Accuracy')
ax2.plot(lstm_history['val_auc'], label='AUC-ROC')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('LSTM+Attention — Validation Metrics')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Model 3: Transformer Encoder

A Transformer encoder uses multi-head self-attention to capture dependencies across all timesteps simultaneously, without the sequential bottleneck of LSTMs.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerClassifier(nn.Module):
    def __init__(self, input_size=20, d_model=64, nhead=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        x = self.input_proj(x)       # (batch, seq, d_model)
        x = self.pos_encoder(x)
        x = self.transformer(x)      # (batch, seq, d_model)
        x = x.mean(dim=1)            # global average pooling
        logits = self.classifier(x).squeeze(-1)
        return logits


model_tf = TransformerClassifier(input_size=N_FEATURES).to(device)
total_params = sum(p.numel() for p in model_tf.parameters())
print(f'Transformer model — Parameters: {total_params:,}')

print('\nTraining Transformer...')
tf_history = train_model(model_tf, train_loader, val_loader, epochs=20, lr=1e-3)

tf_acc, tf_f1, tf_auc = evaluate_model(model_tf, test_loader)
print(f'\nTransformer Test Results:')
print(f'  Accuracy: {tf_acc:.4f}')
print(f'  F1 Score: {tf_f1:.4f}')
print(f'  AUC-ROC:  {tf_auc:.4f}')

## 7. Results Comparison

Side-by-side comparison of all three models on the synthetic test set, plus reference results from real FAIR-MAST data.

In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'Bi-LSTM+Attention', 'Transformer'],
    'Accuracy': [rf_acc, lstm_acc, tf_acc],
    'F1 Score': [rf_f1, lstm_f1, tf_f1],
    'AUC-ROC': [rf_auc, lstm_auc, tf_auc],
})

print('='*65)
print('        DISRUPTION PREDICTION — MODEL COMPARISON')
print('='*65)
print(results.to_string(index=False, float_format='%.4f'))
print('='*65)

print('\n--- Reference: Real FAIR-MAST Results (500 shots, 3-fold CV) ---')
real_results = pd.DataFrame({
    'Model': ['Random Forest', 'Bi-LSTM+Attention', 'Transformer'],
    'CV Accuracy': ['98.4% +/- 0.8%', '78.4% +/- 0.9%', '87.8% +/- 1.0%'],
    'CV AUC-ROC': ['0.983 +/- 0.012', '0.867 +/- 0.013', '0.938 +/- 0.034'],
    'Test AUC': ['1.000', '0.944', '1.000'],
})
print(real_results.to_string(index=False))
print('\nNote: LSTM needs 50+ epochs for optimal performance.')
print('With full training on real data, LSTM+Attention achieves AUC 0.9938.')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
w = 0.25
ax.bar(x - w, results['Accuracy'], w, label='Accuracy', color='#2196F3')
ax.bar(x, results['F1 Score'], w, label='F1 Score', color='#FF9800')
ax.bar(x + w, results['AUC-ROC'], w, label='AUC-ROC', color='#4CAF50')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'])
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Disruption Prediction')
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. Bonus: Time-Series Forecasting

Beyond classification, we can also forecast future plasma signals using a sliding window approach. This is relevant for the [TokaMark benchmark](https://arxiv.org/abs/2602.10132) which evaluates time-series forecasting on MAST data.

We split each shot into past (first 150 timesteps) and future (last 50 timesteps), then train a simple LSTM to predict the future signal from the past.

In [ ]:
PAST_LEN = 150
FUTURE_LEN = 50

class LSTMForecaster(nn.Module):
    def __init__(self, input_size=20, hidden_size=64, output_steps=50):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2,
                           batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, input_size * output_steps)
        self.output_steps = output_steps
        self.input_size = input_size
    
    def forward(self, x):
        _, (h, _) = self.lstm(x)  # use final hidden state
        out = self.fc(h[-1])  # (batch, input_size * output_steps)
        return out.view(-1, self.output_steps, self.input_size)


# Prepare forecasting data
X_past_train = torch.FloatTensor(X_train_norm[:, :PAST_LEN, :])
X_future_train = torch.FloatTensor(X_train_norm[:, PAST_LEN:PAST_LEN+FUTURE_LEN, :])
X_past_test = torch.FloatTensor(X_test_norm[:, :PAST_LEN, :])
X_future_test = torch.FloatTensor(X_test_norm[:, PAST_LEN:PAST_LEN+FUTURE_LEN, :])

forecast_train_ds = TensorDataset(X_past_train, X_future_train)
forecast_test_ds = TensorDataset(X_past_test, X_future_test)
forecast_train_loader = DataLoader(forecast_train_ds, batch_size=32, shuffle=True)
forecast_test_loader = DataLoader(forecast_test_ds, batch_size=32)

forecaster = LSTMForecaster(input_size=N_FEATURES, output_steps=FUTURE_LEN).to(device)
optimizer = optim.Adam(forecaster.parameters(), lr=1e-3)
criterion = nn.MSELoss()

print('Training LSTM Forecaster...')
for epoch in range(20):
    forecaster.train()
    losses = []
    for X_p, X_f in forecast_train_loader:
        X_p, X_f = X_p.to(device), X_f.to(device)
        optimizer.zero_grad()
        pred = forecaster(X_p)
        loss = criterion(pred, X_f)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}/20 — MSE: {np.mean(losses):.6f}')

# Evaluate: NRMSE
forecaster.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_p, X_f in forecast_test_loader:
        X_p = X_p.to(device)
        pred = forecaster(X_p).cpu()
        all_preds.append(pred)
        all_targets.append(X_f)

preds_cat = torch.cat(all_preds).numpy()
targets_cat = torch.cat(all_targets).numpy()

# NRMSE per feature
nrmse_per_feat = []
for f in range(N_FEATURES):
    rmse = np.sqrt(np.mean((preds_cat[:, :, f] - targets_cat[:, :, f]) ** 2))
    data_range = targets_cat[:, :, f].max() - targets_cat[:, :, f].min()
    nrmse = rmse / (data_range + 1e-8)
    nrmse_per_feat.append(nrmse)

avg_nrmse = np.mean(nrmse_per_feat)
print(f'\nForecasting Results (predict {FUTURE_LEN} steps from {PAST_LEN} steps):')
print(f'  Average NRMSE: {avg_nrmse:.4f}')
print(f'  Best feature:  {feature_names[np.argmin(nrmse_per_feat)]} (NRMSE={min(nrmse_per_feat):.4f})')
print(f'  Worst feature: {feature_names[np.argmax(nrmse_per_feat)]} (NRMSE={max(nrmse_per_feat):.4f})')

# Plot example forecast
sample_idx = 0
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, fidx in enumerate([0, 2, 5]):
    ax = axes[col]
    t_past = np.arange(PAST_LEN)
    t_future = np.arange(PAST_LEN, PAST_LEN + FUTURE_LEN)
    ax.plot(t_past, X_test_norm[sample_idx, :PAST_LEN, fidx], 'b-', label='Past (input)')
    ax.plot(t_future, targets_cat[sample_idx, :, fidx], 'g-', linewidth=2, label='True future')
    ax.plot(t_future, preds_cat[sample_idx, :, fidx], 'r--', linewidth=2, label='Predicted')
    ax.axvline(x=PAST_LEN, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(feature_names[fidx])
    ax.set_xlabel('Timestep')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('LSTM Forecaster — Example Predictions', fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusion

We demonstrated three approaches to disruption prediction on tokamak data:

| Approach | Strengths | Limitations |
|----------|-----------|-------------|
| **Random Forest** | Fast training, interpretable features, robust to temporal shift | Loses temporal structure |
| **Bi-LSTM+Attention** | Captures sequential patterns, attention highlights critical moments | Needs longer training, slower convergence |
| **Transformer** | Global self-attention, parallelizable | Higher parameter count, needs more data |

**Key takeaways:**
- On synthetic data, all models can learn the disruption signature (growing oscillations + current drop)
- On real FAIR-MAST data (500 shots), RF achieves the highest quick-mode AUC (0.983), while LSTM+Attention reaches **AUC 0.9938** with full training
- Temporal validation (train on early shots, test on later shots) is essential — random splits inflate metrics
- The forecasting extension enables participation in the [TokaMark benchmark](https://arxiv.org/abs/2602.10132)

**Explore the full platform:**
- Web app: [tokamak-ai.ru](https://tokamak-ai.ru) — load real MAST shots, train models, visualize predictions
- GitHub: [sakiselev-ai/tokamak-analysis](https://github.com/sakiselev-ai/tokamak-analysis) — full source code (MIT license)

---
*Built at НИЯУ МИФИ (National Research Nuclear University MEPhI) | 2026*